In [ ]:
import IPython.display
import dataclasses
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors
import astropy.units as u
import astropy.visualization
import named_arrays as na
import ctis
import esis

In [ ]:
velocity_max = 250 * u.km / u.s

In [ ]:
%%time
scene = esis.flights.f1.data.synth.scene_iris(
    time_start="2014-07-04 11:40",
    velocity_max=velocity_max,
)

In [ ]:
scene.inputs.position = scene.inputs.position - scene.inputs.position.mean()

In [ ]:
scene = scene[{scene.axis_time: 0}]
scene.timedelta = scene.timedelta[{scene.axis_time: 0}].mean(scene.axis_detector_x)

In [ ]:
scene.show();

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        scene.inputs.velocity,
        scene.outputs.mean((scene.axis_detector_x, scene.axis_detector_y)),
        ax=ax,
        axis=scene.axis_wavelength,
    )
    ax.set_xlabel(f"Doppler velocity ({ax.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})");

In [ ]:
wavelength_rest = scene.inputs.wavelength_rest
axis_wavelength = scene.axis_wavelength
axis_xy = (scene.axis_detector_x, scene.axis_detector_y)

In [ ]:
design = esis.flights.f1.optics.design(num_distribution=0)

In [ ]:
design.wavelength = na.linspace(
    start=design.wavelength_min.mean(),
    stop=design.wavelength_max.mean(),
    axis="fit-wavelength",
    num=11,
)

In [ ]:
%%time
system_linear = design.system.linearize()

In [ ]:
axis_channel = design.axis_channel
channel = "Channel " + design.camera.channel.astype(str)

In [ ]:
%%time
images = system_linear.image(scene, chunk_size=8)

In [ ]:
images = np.maximum(images, 0)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = na.plt.subplots(
        axis_rows="rows",
        axis_cols="cols",
        nrows=2,
        ncols=2,
        constrained_layout=True,
        figsize=(10, 9),
        sharex=True,
        sharey=True,
        origin="upper",
    )
    ax = axs.combine_axes(("rows", "cols"), axis_channel)
    norm = matplotlib.colors.PowerNorm(
        gamma=0.5,
        vmin=0,
        vmax=images.outputs.value.percentile(99.99).ndarray,
    )
    colorizer = plt.Colorizer(
        cmap="gray",
        norm=norm,
    )
    img = na.plt.pcolormesh(
        images.inputs.position.x,
        images.inputs.position.y,
        C=images.outputs.value,
        ax=ax,
        colorizer=colorizer,
    )
    plt.colorbar(
        mappable=plt.cm.ScalarMappable(colorizer=colorizer),
        ax=ax.ndarray,
        label=f"signal ({images.outputs.unit:latex_inline})",
    )
    na.plt.set_title(channel, ax=ax)
    na.plt.set_aspect("equal", ax=ax)
    na.plt.set_xlabel(f"sensor $x$ ({images.inputs.position.x.unit})", ax=axs[dict(rows=~0)])
    na.plt.set_ylabel(f"sensor $y$ ({images.inputs.position.y.unit})", ax=axs[dict(cols=0)])
    na.plt.set_xlim(1024 + 256, 1024 + 512 + 256, ax=ax)
    na.plt.set_ylim(256, 512 + 256, ax=ax)

In [ ]:
coefficients = system_linear.distortion.fit.coefficients

In [ ]:
dispersion = 1 / coefficients.components["wavelength"].length.ndarray.mean() * u.pix
dispersion = (wavelength_rest + dispersion).to(u.km / u.s, equivalencies=u.doppler_optical(wavelength_rest)) / u.pix
dispersion

In [ ]:
plate_scale = 1 / coefficients.components["position.x"].length.ndarray.mean()
plate_scale = plate_scale.to(u.arcsec / u.pix)
plate_scale

In [ ]:
velocity_degraded = ctis.arange(
    start=-velocity_max,
    stop=+velocity_max,
    step=dispersion * u.pix,
    axis=scene.axis_wavelength,
)

In [ ]:
position_degraded = ctis.arange(
    start=scene.inputs.position.min(),
    stop=scene.inputs.position.max(),
    step=plate_scale * u.pix,
    axis=na.Cartesian2dVectorArray(
        x=scene.axis_detector_x,
        y=scene.axis_detector_y,
    )
)

In [ ]:
coordinates_degraded = na.TemporalDopplerPositionalVectorArray.from_velocity(
    time=scene.inputs.time.mean(scene.axis_detector_x),
    velocity=velocity_degraded,
    wavelength_rest=wavelength_rest,
    position=position_degraded,
)

In [ ]:
scene_degraded = ctis.regrid(
    coordinates_input=scene.inputs,
    coordinates_output=coordinates_degraded,
    values_input=scene.outputs,
    axis_wavelength=scene.axis_wavelength,
    axis_position=(scene.axis_detector_x, scene.axis_detector_y),
)
scene_degraded = scene.replace(
    inputs=coordinates_degraded,
    outputs=scene_degraded,
)

In [ ]:
scene_degraded.show();

In [ ]:
instrument = ctis.instruments.OptikaInstrument(
    system=system_linear,
    coordinates_scene=coordinates_degraded,
    channel=design.camera.channel,
    axis_channel=design.axis_channel,
    axis_wavelength=scene.axis_wavelength,
    axis_scene_xy=(scene.axis_detector_x, scene.axis_detector_y),
)

In [ ]:
instrument_monochromatic = dataclasses.replace(
    instrument,
    coordinates_scene=coordinates_degraded.replace(wavelength=images.inputs.wavelength),
)

In [ ]:
%%time
backprojected_monochromatic = instrument_monochromatic.backproject(images, unit=scene.outputs.unit)

In [ ]:
min_monochromatic = np.min(backprojected_monochromatic, axis=axis_channel)[{axis_channel: 0}]

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    img = na.plt.pcolormesh(
        min_monochromatic.inputs.position.x,
        min_monochromatic.inputs.position.y,
        C=min_monochromatic.outputs.mean(axis_wavelength).value,
        ax=ax,
        cmap="gray",
        vmin=0,
        vmax=np.percentile(min_monochromatic.outputs, 99.5).value,
    )
    fig.colorbar(
        img.ndarray.item(),
        ax=ax,
        label=f"radiance ({min_monochromatic.outputs.unit:latex_inline})",
    )
    ax.set_aspect("equal")
    ax.set_xlabel(f"scene $x$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"scene $y$ ({ax.get_ylabel()})");

In [ ]:
guess_spatial = min_monochromatic.outputs
guess_spatial = guess_spatial / guess_spatial.sum()

In [ ]:
velocity = scene_degraded.inputs.velocity.cell_centers(axis_wavelength)
width = 56.25 * u.km / u.s
kappa = 0.2
guess_spectral = (1 + np.square(velocity / width) / kappa) ** (-kappa - 1)
guess_spectral = guess_spectral / guess_spectral.sum()

In [ ]:
backprojected = instrument.backproject(images, unit=scene.outputs.unit)
guess_total = backprojected.outputs.sum() / backprojected.shape[axis_channel]

In [ ]:
guess = guess_spectral * guess_spatial * guess_total

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        scene_degraded.inputs.velocity,
        scene_degraded.outputs.mean(axis_xy),
        ax=ax,
        label="degraded scene",
        axis=axis_wavelength,
    )
    na.plt.stairs(
        scene_degraded.inputs.velocity,
        guess.mean(axis_xy),
        ax=ax,
        label="guess",
        axis=axis_wavelength,
    )
    ax.set_xlabel(f"Doppler velocity ({ax.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")
    ax.legend();

In [ ]:
mart = ctis.inverters.MartInverter(
    instrument=instrument,
    intermediate=True,
    num_iteration=50,
)

In [ ]:
%%time
inversion = mart(images, guess=guess, verbose=True)

In [ ]:
solution = inversion.solution

In [ ]:
axis_iter = inversion.inverter.axis_iteration
fig, ax = plt.subplots(
    nrows=2,
    sharex=True,
    constrained_layout=True,
)
na.plt.plot(
    inversion.iteration,
    inversion.mean_chi_squared,
    ax=ax[0],
    axis=axis_iter,
    label=channel,
)
na.plt.plot(
    inversion.iteration,
    inversion.correlation_residual,
    ax=ax[1],
    axis=axis_iter,
    label=channel,
)
ax[0].set_ylabel(r"$\langle \chi^2 \rangle$")
ax[1].set_xlabel("iteration")
ax[1].set_ylabel("signal-correlated residual")
ax[0].set_yscale("log")
ax[0].legend();

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        scene_degraded.inputs.velocity,
        scene_degraded.outputs.mean(axis_xy),
        ax=ax,
        label="degraded scene",
        axis=axis_wavelength,
    )
    na.plt.stairs(
        scene_degraded.inputs.velocity,
        guess.mean(axis_xy),
        ax=ax,
        label="guess",
        axis=axis_wavelength,
    )
    na.plt.stairs(
        solution.inputs.velocity,
        solution.outputs.mean(axis_xy),
        ax=ax,
        label="reconstruction",
        axis=axis_wavelength,
    )
    ax.set_xlabel(f"Doppler velocity ({ax.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")
    ax.legend();

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=3,
        gridspec_kw=dict(width_ratios=[.5, .5, .1]),
        constrained_layout=True,
        figsize=(10, 6),
    )
    ax1, ax2, cax = axs
    ax2.set_yticklabels([])
    vmin = np.nanpercentile(
        a=scene_degraded.outputs,
        q=0.5,
        axis=axis_xy,
    )
    vmax = np.nanpercentile(
        a=scene_degraded.outputs,
        q=99.5,
        axis=axis_xy,
    )
    for ax, cube in zip([ax1, ax2], [scene_degraded, solution]):
        colorbar = na.plt.rgbmesh(
            scene_degraded.inputs.velocity.cell_centers(axis_wavelength),
            scene_degraded.inputs.position.x,
            scene_degraded.inputs.position.y,
            C=cube.outputs,
            axis_wavelength=axis_wavelength,
            ax=ax,
            vmin=vmin,
            vmax=vmax,
            wavelength_min=-velocity_max,
            wavelength_max=+velocity_max,
        )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb=axis_wavelength,
        ax=cax,
    )
    ax1.set_title("degraded scene")
    ax2.set_title("reconstruction")
    unit_x = scene_degraded.inputs.position.x.unit
    unit_y = scene_degraded.inputs.position.y.unit
    ax1.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax2.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax1.set_ylabel(f"scene $y$ ({unit_y:latex_inline})")
    ax2.set_ylabel("")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(
        ncols=2,
        constrained_layout=True,
        sharex=True,
        sharey=True,
        figsize=(6, 5),
    )
    i = {scene.axis_detector_x: 100}
    i0 = {scene.axis_detector_x: 0, scene.axis_detector_y: 0}
    vmax = scene_degraded.outputs.percentile(99.9).ndarray.value
    norm = matplotlib.colors.PowerNorm(gamma=0.5, vmin=0, vmax=vmax)
    na.plt.pcolormesh(
        scene_degraded.inputs.velocity[i0],
        scene_degraded.inputs.position.y[i],
        C=scene_degraded.outputs[i].value,
        ax=ax[0],
        norm=norm,
    )
    ani = na.plt.pcolormovie(
        inversion.iteration,
        scene_degraded.inputs.velocity[i0],
        inversion.solutions.inputs.position.y[i],
        C=inversion.solutions.outputs[i].to_value(scene_degraded.outputs.unit),
        ax=ax[1],
        axis_time=inversion.inverter.axis_iteration,
        norm=norm,
    )
    ax[0].set_title("degraded scene")
    ax[1].set_title("reconstruction")

result = IPython.display.HTML(ani.to_jshtml(fps=10))
plt.close(ani._fig)
result

In [ ]:
inversion.plot_moments(
    truth=scene_degraded,
    axis=axis_wavelength,
    range_radiance=[0, 3e3] * u.erg / u.s / u.sr / u.cm**2,
    range_median=[-60, 60] * u.km / u.s,
    range_iqr=[0, 100] * u.km / u.s,
    percentile_radiance=75,
);

In [ ]:
%%time
predictions = instrument.image(solution, noise=False)

In [ ]:
residual = images - predictions

In [ ]:
fig, axs = na.plt.subplots(
    axis_rows="rows",
    axis_cols="cols",
    nrows=2,
    ncols=2,
    constrained_layout=True,
    figsize=(8, 7),
    sharex=True,
    sharey=True,
    origin="upper",
)
ax = axs.combine_axes(("rows", "cols"), axis_channel)
img = na.plt.pcolormesh(
    residual.inputs.position.x,
    residual.inputs.position.y,
    C=residual.outputs.value,
    ax=ax,
    vmin=residual.outputs.value.percentile(0.01).ndarray,
    vmax=residual.outputs.value.percentile(99.99).ndarray,
    cmap="gray",
)
na.plt.set_aspect("equal", ax=ax)
plt.colorbar(
    ax=ax.ndarray,
    mappable=img.ndarray[0],
    label=f"residual ({residual.outputs.unit:latex_inline})",
)
na.plt.set_title(channel, ax=ax)
na.plt.set_xlabel(f"sensor $x$ ({residual.inputs.position.x.unit})", ax=axs[dict(rows=~0)])
na.plt.set_ylabel(f"sensor $y$ ({residual.inputs.position.y.unit})", ax=axs[dict(cols=0)])
na.plt.set_xlim(1024 + 256, 1024 + 512 + 256, ax=ax)
na.plt.set_ylim(256, 512 + 256, ax=ax);